# ICU Step-Down Readiness — execution evidence

Committed notebook with **cell outputs** reproducing the live run against the
Databricks sandbox `fe-sandbox-icu-step-down-readiness` (profile `icu-sandbox`).
The outputs below are the genuine captured results; raw captures are in
`../evidence/raw/` and `./` (see `logs/README.md`).

## 1. Lakeflow pipeline run — bronze → silver → gold row counts

In [1]:
# databricks pipelines start-update icu-step-down-readiness-medallion  -> state COMPLETED
spark.sql('''
  SELECT 'gold' lyr,'census' tbl, COUNT(*) rows FROM icu_step_down.gold.census
  UNION ALL SELECT 'gold','patient_features', COUNT(*) FROM icu_step_down.gold.patient_features
  UNION ALL SELECT 'bronze','chart_events', COUNT(*) FROM icu_step_down.bronze.chart_events
''').show()

Update a28ac017 state: COMPLETED
+------+----------------+--------+
|   lyr|             tbl|    rows|
+------+----------------+--------+
|  gold|          census|      40|
|  gold|patient_features|   61532|
|bronze|    chart_events|38776289|
+------+----------------+--------+
bronze.admissions=58976  bronze.lab_events=27854055  silver.vital_signs=21771413


## 2. Query against the governed gold tables

In [2]:
spark.sql('''SELECT readiness_label, COUNT(*) n,
  ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(),1) pct
  FROM icu_step_down.gold.readiness_training_set GROUP BY readiness_label''').show()

+---------------+-----+----+
|readiness_label|    n| pct|
+---------------+-----+----+
|              0| 1826| 3.0|
|              1|59706|97.0|
+---------------+-----+----+
# current census cohort: patients=40 avg_age=54.8 avg_los_days=2.83 on_pressors=8 on_vent=11


## 3. Real prediction from the Mosaic AI serving endpoint `icu-readiness`

In [3]:
w.serving_endpoints.query('icu-readiness', dataframe_records=records)  # model readiness_model v2

Patient A (stable):   readiness_score=0.4739  factors=[lactate_last risk 0.262, on_ventilator risk 0.030, rr_last supports 0.029]
Patient B (critical): readiness_score=0.4632  factors=[lactate_last risk 0.200, rr_min risk 0.110, hr_mean supports 0.080]
# on_ventilator forced to "risk" — always-flag-known-risks safety override, live


## 4. Genie natural-language Q&A over the governed tables

In [4]:
genie.ask('How many current ICU patients are ready for step-down?')  # -> generated SQL -> rows

generated SQL: SELECT SUM(CASE WHEN r.readiness_label=1 THEN 1 ELSE 0 END) ready,
  SUM(CASE WHEN r.readiness_label=0 THEN 1 ELSE 0 END) not_ready, COUNT(*) total
  FROM icu_step_down.gold.census c JOIN icu_step_down.gold.readiness_training_set r USING(icustay_id)
rows: ready_for_step_down=38 | not_ready=2 | total_current_patients=40


## 5. Model drift job (live vs training baseline)

In [5]:
spark.sql('SELECT metric, value, verdict, n_live FROM icu_step_down.gold.readiness_drift').show()

+------+------+-------+------+
|metric| value|verdict|n_live|
+------+------+-------+------+
|   PSI|2.1528|  drift|    40|
|    KS|0.2205|  drift|    40|
+------+------+-------+------+


## 6. Deployed app — RUNNING

In [6]:
w.apps.get('icu-step-down')

app_state=RUNNING compute_state=ACTIVE
url=https://icu-step-down-7474645692590282.aws.databricksapps.com
[APP] INFO: Application startup complete.
